# Smart Waste Management - Kaggle Training

Before running this notebook, enable **Internet** and select a **GPU T4** accelerator in Kaggle notebook settings. The final `waste_model.onnx` and `class_names.json` files are copied to `/kaggle/working/` for download.

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/aadyaraikar/smart_waste_management.git"
REPO_ROOT = Path("/kaggle/working/smart_waste_management")

if not REPO_ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
else:
    print(f"Using existing repository: {REPO_ROOT}")

%cd /kaggle/working/smart_waste_management
print(f"Repository: {Path.cwd()}")

In [ ]:
%pip install -q tf2onnx

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
print("Available GPUs:", gpus)
if not gpus:
    raise RuntimeError("No GPU detected. Enable a Kaggle T4 accelerator before training.")

In [ ]:
import sys
import subprocess
from pathlib import Path

class_names = [
    "E_Waste_Batteries",
    "Glass",
    "Metal",
    "Organic_Wet",
    "Paper_Cardboard",
    "Plastic",
    "Textiles_Footwear",
]
dataset_root = Path("ML/dataset")
dataset_ready = all((dataset_root / class_name).is_dir() for class_name in class_names)

if not dataset_ready:
    subprocess.run([sys.executable, "ML/prepare_dataset.py"], check=True)
else:
    print(f"Using existing dataset at {dataset_root}")

In [ ]:
subprocess.run([sys.executable, "ML/train.py"], check=True)

In [ ]:
import json
import shutil
from pathlib import Path

model_path = Path("ML/waste_model.onnx")
if not model_path.exists():
    raise FileNotFoundError(f"Training did not produce {model_path}")

class_names_path = Path("ML/class_names.json")
class_names_path.write_text(json.dumps(class_names, indent=2) + "\n", encoding="utf-8")

output_model = Path("/kaggle/working/waste_model.onnx")
output_classes = Path("/kaggle/working/class_names.json")
shutil.copy2(model_path, output_model)
shutil.copy2(class_names_path, output_classes)

print(f"Model output: {output_model} ({output_model.stat().st_size:,} bytes)")
print(f"Class manifest: {output_classes}")